In [1]:
import torch


print(f"PyTorch version {torch.__version__}")

if torch.cuda.is_available():
    print(f"CUDA/ROCm GPU: {torch.cuda.get_device_name(0)}")

elif torch.xpu.is_available():
    print(f"Intel GPU: {torch.xpu.get_device_name(0)}")

elif torch.backends.mps.is_available():
    print("Apple Silicon GPU")

else:
    print("Only CPU")

PyTorch version 2.11.0+cu128
CUDA/ROCm GPU: Tesla T4


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

model_name = "HuggingFaceTB/SmolLM-360M"

print(f"Loading tokenizer for {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=HF_TOKEN
)

print(f"Loading model for {model_name}...")

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    token=HF_TOKEN
)

Loading tokenizer for HuggingFaceTB/SmolLM-360M...


config.json:   0%|          | 0.00/725 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.69k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Loading model for HuggingFaceTB/SmolLM-360M...


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.45GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

## Tokenize

In [3]:
prompt = "What is llm"
inputs = tokenizer(prompt, return_tensors="pt")

print(f"Original prompt: {prompt}")
print(f"Tokenized input IDs: {inputs['input_ids']}")
print(f"Decoded tokens: {tokenizer.decode(inputs['input_ids'][0])}")

Original prompt: What is llm
Tokenized input IDs: tensor([[ 1780,   314,   303, 31431]])
Decoded tokens: What is llm


## Generate a small prompt follow up

In [4]:
def generate_response(model, tokenizer, prompt_text, max_new_tokens=50):
    inputs = tokenizer(prompt_text, return_tensors="pt")
    # Check if a GPU is available and move inputs to GPU if it is
    if torch.cuda.is_available():
        inputs = {k: v.to('cuda') for k, v in inputs.items()}
        model.to('cuda')

    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, num_return_sequences=1)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

# Generate a response using the loaded model
generated_text = generate_response(model, tokenizer, prompt)
print(f"Generated response: {generated_text}")

Generated response: What is llm?
The LLM is a postgraduate qualification that is designed to provide students with a broad and deep understanding of the subject. It is a highly respected qualification that is recognised by employers and universities around the world.
What is the difference between a L


## Benchmark response between CPU and GPU

In [5]:
import time

prompt = "What is llm"

def benchmark_generation(model, tokenizer, prompt_text, device, max_new_tokens=50):
    model.to(device)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(device)

    start_time = time.time()
    _ = model.generate(**inputs, max_new_tokens=max_new_tokens, num_return_sequences=1)
    end_time = time.time()
    return end_time - start_time

# Benchmark on CPU
print("Benchmarking on CPU...")
cpu_time = benchmark_generation(model, tokenizer, prompt, 'cpu')
print(f"CPU generation time: {cpu_time:.4f} seconds")

# Benchmark on GPU if available
if torch.cuda.is_available():
    print("Benchmarking on GPU...")
    gpu_time = benchmark_generation(model, tokenizer, prompt, 'cuda')
    print(f"GPU generation time: {gpu_time:.4f} seconds")
else:
    print("GPU not available for benchmarking.")

Benchmarking on CPU...
CPU generation time: 12.4712 seconds
Benchmarking on GPU...
GPU generation time: 3.2389 seconds


We can run the same prompt in two modes:

Normal generation → generation stops when EOS is produced.
Ignore EOS → generation continues even after EOS, and we visually mark the EOS token(s) in a different color.

One subtlety: model.generate() normally doesn't return tokens after EOS because EOS is a stopping criterion. So for experiment #2 we need to disable EOS-based stopping.

First, let's create a visualization helper

Since you're using Colab, we can render the EOS token in red.

In [6]:
from IPython.display import display, HTML
import torch
import time


def show_tokens_with_eos(tokenizer, token_ids):
    tokens = tokenizer.convert_ids_to_tokens(token_ids)

    html = ""

    for token in tokens:
        # Decode this individual token for display
        text = tokenizer.convert_tokens_to_string([token])

        # Identify EOS
        if token == tokenizer.eos_token_id:
            html += (
                f'<span style="color:red; font-weight:bold; '
                f'background-color:#ffe6e6; padding:2px 4px; '
                f'border-radius:3px;">'
                f'{text if text else "[EOS]"}'
                f'</span>'
            )
        else:
            # Preserve spaces in HTML
            text = text.replace(" ", "&nbsp;")
            html += text

    display(HTML(f"<div style='font-size:16px'>{html}</div>"))

1. Normal generation — stop at EOS

Here we let Transformers handle EOS normally.

In [14]:
def generate_until_eos(model, tokenizer, prompt_text, device, max_new_tokens=50):

    model.to(device)

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt"
    ).to(device)

    start_time = time.time()

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_return_sequences=1,
            eos_token_id=tokenizer.eos_token_id
        )

    if device == "cuda":
        torch.cuda.synchronize()

    end_time = time.time()

    return output_ids[0], end_time - start_time

In [15]:
from transformers import LogitsProcessor


class ForceEOS(LogitsProcessor):
    def __init__(self, eos_token_id, force_after):
        self.eos_token_id = eos_token_id
        self.force_after = force_after

    def __call__(self, input_ids, scores):
        # input_ids contains prompt + generated tokens
        if input_ids.shape[1] >= self.force_after:
            scores[:, :] = float("-inf")
            scores[:, self.eos_token_id] = 0

        return scores

2. Ignore EOS and continue generating

Now we deliberately tell generate():

Don't stop when you encounter EOS.

In [16]:
eos_token_id=None

def generate_ignore_eos(model, tokenizer, prompt_text, device, max_new_tokens=50):

    model.to(device)

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt"
    ).to(device)

    start_time = time.time()

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_return_sequences=1,

            # IMPORTANT:
            # Don't stop when EOS is generated
            eos_token_id=None
        )

    if device == "cuda":
        torch.cuda.synchronize()

    end_time = time.time()

    return output_ids[0], end_time - start_time

In [17]:
output_ids_no_stop, generation_time = generate_ignore_eos(
    model,
    tokenizer,
    prompt,
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Generation time: {generation_time:.4f} seconds")

print("\nGenerated text:")
print(tokenizer.decode(output_ids_no_stop, skip_special_tokens=False))

print("\nToken visualization:")
show_tokens_with_eos(tokenizer, output_ids_no_stop)

Generation time: 4.0059 seconds

Generated text:
What is llm?
The LLM is a postgraduate qualification that is designed to provide students with a broad and deep understanding of the subject. It is a highly respected qualification that is recognised by employers and universities around the world.
What is the difference between a L

Token visualization:
